# Figure 3 – Material classification and data sufficiency

Data are read from `data/` (written by `run_all.py`). Every figure is saved as `Figure3_<nn>_<title>.png` when it is shown.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

DATA = Path("data")
PREFIX = "Figure3"
_plt_show = plt.show
_n = [0]


def _title(fig):
    for ax in fig.axes:
        t = ax.get_title()
        if t:
            return re.sub(r"[^A-Za-z0-9]+", "_", t).strip("_")[:50]
    return "figure"


def _save_then_show(*args, **kwargs):
    for num in plt.get_fignums():
        fig = plt.figure(num)
        _n[0] += 1
        name = f"{PREFIX}_{_n[0]:02d}_{_title(fig)}.png"
        fig.savefig(name, dpi=fig.dpi, bbox_inches="tight")
        print("saved:", name)
    _plt_show(*args, **kwargs)


plt.show = _save_then_show

In [ ]:
from types import SimpleNamespace
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

pred = pd.read_csv(DATA / "material_predictions.csv")
y_mat_true = pred["y_mat_true"].values
y_mat_pred = pred["y_mat_pred"].values
le_mat = SimpleNamespace(classes_=pd.read_csv(DATA / "classes.csv")["le_mat_classes_"].values)

def load_H(name):
    H = pd.read_csv(DATA / name, index_col=0)
    return H.to_numpy(float)

sample_ratios = [0.1, 0.2, 0.3,0.4, 0.5,0.6, 0.7,0.8, 0.9,1.0]
pulse_nums = [1, 2, 3, 4, 5]

## Confusion matrix of material classification (test batteries)

In [ ]:
print("Material classification accuracy:",
      accuracy_score(y_mat_true, y_mat_pred))

cm = confusion_matrix(
    y_mat_true,
    y_mat_pred,
    labels=le_mat.classes_
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le_mat.classes_
)

fig, ax = plt.subplots(figsize=(3, 2),dpi=600)
disp.plot(
    ax=ax,
    cmap="Blues",
    colorbar=False,
    values_format="d"
)

ax.set_title("Material Classification Confusion Matrix\n(New Batteries)",
             fontsize=10)
ax.set_xlabel("Predicted Material", fontsize=9)
ax.set_ylabel("True Material", fontsize=9)

plt.tight_layout()
plt.show()

## Confusion matrix of material classification (frameless style)

In [ ]:
print("Material classification accuracy:",
      accuracy_score(y_mat_true, y_mat_pred))

cm = confusion_matrix(
    y_mat_true,
    y_mat_pred,
    labels=le_mat.classes_
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le_mat.classes_
)

fig, ax = plt.subplots(figsize=(3, 2), dpi=600)

disp.plot(
    ax=ax,
    cmap="Blues",
    colorbar=False,
    values_format="d"

)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title(
    "Material Classification Confusion Matrix\n(New Batteries)",
    fontsize=10
)
ax.set_xlabel("Predicted Material", fontsize=9)
ax.set_ylabel("True Material", fontsize=9)

plt.tight_layout()
plt.show()

## Data sufficiency: number of pulses × training sample ratio

Each cell is the mean of three repeats (seeds 0, 1, 2).

In [ ]:
H_soh = load_H("pulse_count_H_soh.csv")
H_soc = load_H("pulse_count_H_soc.csv")
H_mat = load_H("pulse_count_H_mat.csv")

fig, axes = plt.subplots(1, 3, figsize=(10, 3), dpi=300, sharey=True)

titles = ["SOH MAE", "SOC MAE", "Material Accuracy"]
maps = [H_soh, H_soc, H_mat]
cmaps = ["viridis", "viridis", "plasma"]

for ax, M, title, cmap in zip(axes, maps, titles, cmaps):
    im = ax.imshow(M, aspect="auto", origin="lower", cmap=cmap)
    ax.set_title(title)
    ax.set_xticks(range(len(sample_ratios)))
    ax.set_xticklabels(sample_ratios)
    ax.set_xlabel("Training sample ratio")
    fig.colorbar(im, ax=ax, fraction=0.046)

axes[0].set_yticks(range(len(pulse_nums)))
axes[0].set_yticklabels(pulse_nums)
axes[0].set_ylabel("Number of pulses")

plt.tight_layout()
plt.show()

## Mean ± SD across numbers of pulses for each training sample ratio

In [ ]:
import numpy as np
import pandas as pd


mat_mean_across_pulses = np.mean(H_mat, axis=0)
mat_std_across_pulses  = np.std(H_mat, axis=0, ddof=1)

soc_mean_across_pulses = np.mean(H_soc, axis=0)
soc_std_across_pulses  = np.std(H_soc, axis=0, ddof=1)

soh_mean_across_pulses = np.mean(H_soh, axis=0)
soh_std_across_pulses  = np.std(H_soh, axis=0, ddof=1)


df_summary = pd.DataFrame({
    "Training sample ratio": sample_ratios,

    "Material Accuracy Mean": mat_mean_across_pulses,
    "Material Accuracy Std": mat_std_across_pulses,

    "SOC MAE Mean": soc_mean_across_pulses,
    "SOC MAE Std": soc_std_across_pulses,

    "SOH MAE Mean": soh_mean_across_pulses,
    "SOH MAE Std": soh_std_across_pulses,
})

print("\nMean and standard deviation across pulse numbers:")
print(df_summary.to_string(index=False))


df_summary_display = pd.DataFrame({
    "Training sample ratio": sample_ratios,

    "Material Accuracy":
        [f"{m:.4f} ± {s:.4f}"
         for m, s in zip(mat_mean_across_pulses,
                         mat_std_across_pulses)],

    "SOC MAE":
        [f"{m:.4f} ± {s:.4f}"
         for m, s in zip(soc_mean_across_pulses,
                         soc_std_across_pulses)],

    "SOH MAE":
        [f"{m:.4f} ± {s:.4f}"
         for m, s in zip(soh_mean_across_pulses,
                         soh_std_across_pulses)]
})

print("\nPublication-style table:")
print(df_summary_display.to_string(index=False))

## Minimal data requirement found by Q-learning

The state values *V* of each task are stored in `data/rl_V_*.csv`.

In [ ]:
rl = pd.read_csv(DATA / "rl_optimum.csv")
print("Optimal data sufficiency per task")
print("----------------------------------")
for _, r in rl.iterrows():
    print(f"{r['Task']:4s} : pulse={r['pulse_num']}, ratio={r['train_ratio']}")
rl

## Data sufficiency: positive/negative pulse combinations × training sample ratio

In [ ]:
POS_COUNTS = [0, 1, 2, 3]
NEG_COUNTS = [0, 1, 2]

COMBOS = []
for np_ in POS_COUNTS:
    for nn_ in NEG_COUNTS:
        if np_ == 0 and nn_ == 0:
            continue
        COMBOS.append((np_, nn_))

H_soh = load_H("pos_neg_H_soh.csv")
H_soc = load_H("pos_neg_H_soc.csv")
H_mat = load_H("pos_neg_H_mat.csv")

y_labels = [f"+{p} / -{n}" for p, n in COMBOS]

fig, axes = plt.subplots(1, 3, figsize=(11, 4), dpi=300, sharey=True)

titles = ["SOH MAE", "SOC MAE", "Material Accuracy"]
maps = [H_soh, H_soc, H_mat]
cmaps = ["viridis", "viridis", "plasma"]

for ax, M, title, cmap in zip(axes, maps, titles, cmaps):
    im = ax.imshow(M, aspect="auto", origin="lower", cmap=cmap)
    ax.set_title(title)
    ax.set_xticks(range(len(sample_ratios)))
    ax.set_xticklabels(sample_ratios)
    ax.set_xlabel("Training sample ratio")
    fig.colorbar(im, ax=ax, fraction=0.046)

axes[0].set_yticks(range(len(y_labels)))
axes[0].set_yticklabels(y_labels)
axes[0].set_ylabel("Pulse combination (+ / −)")

plt.tight_layout()
plt.show()

## Mean ± SD across pulse combinations for each training sample ratio

In [ ]:
import numpy as np
import pandas as pd


mat_mean_across_combos = np.mean(H_mat, axis=0)
mat_std_across_combos  = np.std(H_mat, axis=0, ddof=1)

soc_mean_across_combos = np.mean(H_soc, axis=0)
soc_std_across_combos  = np.std(H_soc, axis=0, ddof=1)

soh_mean_across_combos = np.mean(H_soh, axis=0)
soh_std_across_combos  = np.std(H_soh, axis=0, ddof=1)


df_summary = pd.DataFrame({
    "Training sample ratio": sample_ratios,

    "Material Accuracy Mean": mat_mean_across_combos,
    "Material Accuracy Std": mat_std_across_combos,

    "SOC MAE Mean": soc_mean_across_combos,
    "SOC MAE Std": soc_std_across_combos,

    "SOH MAE Mean": soh_mean_across_combos,
    "SOH MAE Std": soh_std_across_combos,
})

print("\nMean and standard deviation across pulse combinations:")
print(df_summary.to_string(index=False))


df_summary_display = pd.DataFrame({
    "Training sample ratio": sample_ratios,

    "Material Accuracy":
        [
            f"{m:.4f} ± {s:.4f}"
            for m, s in zip(
                mat_mean_across_combos,
                mat_std_across_combos
            )
        ],

    "SOC MAE":
        [
            f"{m:.4f} ± {s:.4f}"
            for m, s in zip(
                soc_mean_across_combos,
                soc_std_across_combos
            )
        ],

    "SOH MAE":
        [
            f"{m:.4f} ± {s:.4f}"
            for m, s in zip(
                soh_mean_across_combos,
                soh_std_across_combos
            )
        ],
})

print("\nPublication-style table:")
print(df_summary_display.to_string(index=False))

## Feature-space data density

`data/df_res.csv`: for each number of pulse blocks *k* and battery-level training ratio, the mean kNN density of the test samples with respect to the training samples and the SOH MAE / R² (true material and true SOC).

In [ ]:
import seaborn as sns

df_res = pd.read_csv(DATA / "df_res.csv")

sns.set_style("white")
df_res

## Mean kNN density of the test samples (Figure 3d)

In [ ]:
df_res = pd.DataFrame(df_res)

BLOCKS = sorted(df_res["n_blocks"].unique())
RATIOS = sorted(df_res["train_ratio"].unique())

H_density = np.full((len(BLOCKS), len(RATIOS)), np.nan)

for i, b in enumerate(BLOCKS):
    for j, r in enumerate(RATIOS):
        v = df_res.loc[
            (df_res["n_blocks"] == b) &
            (df_res["train_ratio"] == r),
            "mean_density"
        ]
        if len(v) > 0:
            H_density[i, j] = v.values[0]

fig, ax = plt.subplots(figsize=(4, 3), dpi=600)

im = ax.imshow(
    H_density,
    origin="lower",
    aspect="auto",
    cmap="cividis"
)

ax.set_xticks(np.arange(len(RATIOS)))
ax.set_xticklabels([f"{r:.1f}" for r in RATIOS])
ax.set_xlabel("Training sample ratio")

ax.set_yticks(np.arange(len(BLOCKS)))
ax.set_yticklabels(BLOCKS)
ax.set_ylabel("Number of pulse blocks")

ax.set_title("Feature-space data density (mean KNN)")

cbar = fig.colorbar(im, ax=ax, fraction=0.046)
cbar.set_label("Mean KNN density")

plt.tight_layout()
plt.show()